# 01 · Data preparation

**What this does** (all outputs go to `DRIVE_ROOT`, every step is resumable and skips work already done):

1. Downloads the Arabic XL-Sum archive for a **pinned dataset commit**, verifies its SHA256.
2. Keeps the official train / validation / test splits; drops near-empty rows from *train*;
   removes from *train* every exact or near duplicate (MinHash, Jaccard ≥ 0.8) of another document.
   Validation/test are never modified, duplicates there are only flagged.
3. Builds the **internal benchmark scaffold**: news candidates drawn from XL-Sum *test*
   (300–1000 words, stratified by length, excluding any article involved in a duplicate pair),
   plus empty templates for organizational reports and human reference summaries.
4. Validates the benchmark and checks for **leakage** against everything that can influence the model
   (XL-Sum train + validation): exact hash, URL and near-duplicate matches.
5. Writes `data/processed/data_manifest.json` (dataset version, license, download date, file hashes,
   split counts, dedup stats, benchmark file hashes, library versions).

Reference summaries are **never** generated by a model; see `docs/reference_guideline.md`.

Runtime: CPU runtime is enough (no GPU needed). About 5–10 minutes on first run.

In [ ]:
# ==== 0. Settings, pinned install, environment ==========================================
REPO_URL   = "https://github.com/assaabriiii/Offline-Arabic-Text-Summarization-Module.git"
REPO_REF   = "main"     # branch or commit SHA; use a SHA for a fully reproducible run
CONFIGS    = ["configs/default.yaml"]          # later files override earlier ones
DRIVE_ROOT = "/content/drive/MyDrive/arabic_summ"
REQUIREMENTS = "requirements/colab-data.txt"

import importlib, os, subprocess, sys
REPO_DIR = "/content/arabic_summ_repo"

def sh(cmd, secret=None):
    shown = cmd.replace(secret, "***") if secret else cmd
    print("$", shown)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError((r.stdout + r.stderr).replace(secret, "***") if secret else r.stdout + r.stderr)

token = None
try:  # private repo: add a Colab secret named GITHUB_TOKEN (key icon in the left bar)
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
if not os.path.isdir(REPO_DIR):
    sh(f"git clone -q {url} {REPO_DIR}", token)
sh(f"git -C {REPO_DIR} fetch -q {url} {REPO_REF} && git -C {REPO_DIR} checkout -q --detach FETCH_HEAD", token)
sh(f"pip install -q -r {REPO_DIR}/{REQUIREMENTS}")
sh(f"pip install -q --no-deps -e {REPO_DIR}")
sys.path.insert(0, f"{REPO_DIR}/src"); importlib.invalidate_caches()
# a reattached runtime may still hold an older copy of the package in memory: drop it
for _m in [m for m in sys.modules if m == "arabic_summ" or m.startswith("arabic_summ.")]:
    del sys.modules[_m]
os.chdir(REPO_DIR)

from arabic_summ.env import environment_report, format_report
print(format_report(environment_report()))

In [ ]:
# ==== 1. Mount Drive, load config, record environment ===================================
from google.colab import drive
drive.mount("/content/drive")

from arabic_summ.config import load_config, config_hash
from arabic_summ.env import save_environment
cfg = load_config(*CONFIGS, overrides={"paths": {"root": DRIVE_ROOT}})
save_environment(cfg, "01_data")
print("artifact root:", cfg["paths"]["root"], "| config hash:", config_hash(cfg))

In [ ]:
# ==== 2. Unit tests (CPU, a few seconds) ================================================
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/"],
                   capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "unit tests failed - stop here and paste the output"

In [ ]:
# ==== 3. Download XL-Sum (pinned revision, SHA256-verified) =============================
from arabic_summ.data import prepare
m = prepare.step_download(cfg)
print({k: m[k] for k in ("repo_id", "revision", "archive_sha256", "archive_bytes", "license", "download_date_utc")})

In [ ]:
# ==== 4. Quality filter + exact / near-duplicate removal ================================
import pandas as pd
m = prepare.step_dedup(cfg)
display(pd.DataFrame(m["splits"]).T)

In [ ]:
# ==== 5. Length statistics (why XL-Sum references alone don't fit our 10-30 % spec) =====
rows = []
for split in ("train", "validation", "test"):
    df = prepare.load_processed(cfg, split)
    ratio = df.summary_words / df.text_words
    in_range = df.text_words.between(cfg["benchmark"]["input_min_words"], cfg["benchmark"]["input_max_words"])
    rows.append({"split": split, "docs": len(df),
                 "text_words_median": df.text_words.median(), "text_words_p90": df.text_words.quantile(.9),
                 "summary_words_median": df.summary_words.median(),
                 "summary/text_ratio_median": round(ratio.median(), 3),
                 "share_300_1000_words": round(in_range.mean(), 3),
                 "share_summary_>=30_words": round((df.summary_words >= 30).mean(), 3)})
display(pd.DataFrame(rows).set_index("split"))

In [ ]:
# ==== 6. Internal benchmark scaffold ====================================================
m = prepare.step_benchmark_scaffold(cfg)
print({k: m[k] for k in ("written", "kept_existing", "news_candidates", "news_by_length_bin", "test_articles_in_range")})

### Benchmark items (rerun cells 7–9 any time; they never overwrite reviewed rows)

* **News** (cell 6): 150 XL-Sum *test* candidates.
* **Organizational reports** (cell 7): the repo ships `benchmark_sources/org_reports.csv`, a list of public
  Arabic pages from WHO EMRO, UNICEF MENA, UNFPA and UN Women Arab States (URLs only; texts are copyrighted and
  are fetched here for non-commercial research evaluation, never committed). You can also drop your own
  UTF-8 `.txt` reports into `DRIVE_ROOT/data/benchmark/incoming_org/` with a `sources.csv` (same columns as
  `external_sources_template.csv`).
* **Selection** (cell 8): automatic, seeded and documented: screening (300–1000 words, ≥ 90 % Arabic script,
  ≤ 15 % numeric tokens), then 120 per domain spread evenly over length bins and round-robin over sources.
  Unsampled items are kept as `rejected` + note `reserve`; promote one to `selected` to replace a bad pick.

**Reference summaries are not automated.** Open `references.csv` in Google Sheets and fill only the rows whose
`sample_id` is `selected` in `benchmark_items.csv`, following `docs/reference_guideline.md`
(writer + a different reviewer, `review_status=approved`).

In [ ]:
# ==== 7. Organizational reports: fetch the committed URL list + import manual uploads ===
from arabic_summ import config as C
from arabic_summ.data import benchmark as B
rep = prepare.fetch_org_reports(cfg, REPO_DIR)          # resumable: skips files already on Drive
print(rep)
manual = C.path(cfg, "benchmark") / "incoming_org"
manual.mkdir(parents=True, exist_ok=True)
src_csv = manual / "sources.csv"
added = B.import_text_folder(C.path(cfg, "benchmark"), manual, "org_report", cfg["benchmark"],
                             src_csv if src_csv.exists() else None)
print(f"manual uploads imported: {len(added)}")

In [ ]:
# ==== 8. Automatic, seeded item selection (120 per domain) ==============================
import json
print(json.dumps(prepare.run_auto_select(cfg), ensure_ascii=False, indent=1))

In [ ]:
# ==== 9. Validate benchmark + leakage check (STRICT=True only when freezing the set) ====
STRICT = False
rep, leak = prepare.validate_benchmark(cfg, strict=STRICT)
print(rep.summary())
print(f"leakage vs XL-Sum train+validation: {leak['n_leaks']} of {leak['items_checked']} items")

In [ ]:
# ==== 10. Manifest summary (paste this output back) ======================================
from arabic_summ.io_utils import read_json
man = read_json(prepare.manifest_path(cfg))
print("dataset :", {k: man["dataset"][k] for k in ("repo_id", "revision", "archive_sha256", "license", "download_date_utc")})
print("files   :", {k: (v["rows"], v["sha256"][:12]) for k, v in man["source_files"].items()})
print("splits  :", {k: (v["raw_rows"], v["kept_rows"], v["matches_expected"]) for k, v in man["splits"].items()})
print("dedup   :", man["dedup"]["stats"])
print("bench   :", man["benchmark"]["validation"]["stats"], "| leakage ok:", man["benchmark"]["leakage"]["ok"])
print("code    :", man["code"])